In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
full_df = pd.read_csv("SAML-D.csv")

In [4]:
full_df.head(2)

,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
0,10:35:19,2022-10-07,8724731955,2769355426,1459.15,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits
1,10:35:20,2022-10-07,1491989064,8401255335,6019.64,UK pounds,Dirham,UK,UAE,Cross-border,0,Normal_Fan_Out


In [5]:
full_df.shape

(9504852, 12)

In [6]:
full_df.isnull().sum()

Time                      0
Date                      0
Sender_account            0
Receiver_account          0
Amount                    0
Payment_currency          0
Received_currency         0
Sender_bank_location      0
Receiver_bank_location    0
Payment_type              0
Is_laundering             0
Laundering_type           0
dtype: int64

In [7]:
full_df.duplicated().sum()

np.int64(0)

In [8]:
full_df['Is_laundering'].value_counts()

Is_laundering
0    9494979
1       9873
Name: count, dtype: int64

## Resampling to balance the imbalance data

In [9]:
df_fraud = full_df[full_df['Is_laundering']==1]

In [10]:
df_fraud.shape

(9873, 12)

In [11]:
df_fraud.shape[0]

9873

In [12]:
df_non_fraud = full_df[full_df['Is_laundering']==0]

### We need to take a sample of exactly 9873 rows from 'non-fraud' data

In [13]:
df_non_fraud_sample = df_non_fraud.sample(n=df_fraud.shape[0], random_state=42)

### Combining the datasets

In [14]:
df_combined = pd.concat([df_fraud, df_non_fraud_sample], axis=0)

In [15]:
df_combined.shape

(19746, 12)

### Shuffling the dataset so the dataset is mixed randomly thoroughly

In [16]:
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

In [17]:
df_combined.head(2)

,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
0,07:40:08,2023-07-14,3187084517,8846937355,3788.78,UK pounds,UK pounds,UK,UK,Cash Deposit,1,Smurfing
1,22:04:14,2022-10-29,4851027632,9832274459,7377.62,UK pounds,Euro,UK,France,Cross-border,0,Normal_Fan_Out


In [18]:
df_combined_tab = pd.crosstab(index=df_combined['Laundering_type'],
                              columns=df_combined['Is_laundering']
                             )
df_combined_tab

Is_laundering,0,1
Laundering_type,,
Behavioural_Change_1,0,394
Behavioural_Change_2,0,345
Bipartite,0,383
Cash_Withdrawal,0,1334
Cycle,0,382
Deposit-Send,0,945
Fan_In,0,364
Fan_Out,0,237
Gather-Scatter,0,354


In [19]:
##### So we cee that the "non-fraud" transactions often start with the word "Normal..." in the column "Laundering type"
##### including this would cause "Data Leakage" where the model would be predicting perfectly but wont be able to 
##### predict anything if this column is absent. In real world, we actually do not get this column. Hence, we would be removing
##### this column to prevent "Data Leakage".

In [20]:
df_combined = df_combined.drop('Laundering_type', axis=1)

In [21]:
df_combined.head(2)

,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering
0,07:40:08,2023-07-14,3187084517,8846937355,3788.78,UK pounds,UK pounds,UK,UK,Cash Deposit,1
1,22:04:14,2022-10-29,4851027632,9832274459,7377.62,UK pounds,Euro,UK,France,Cross-border,0


In [22]:
df_combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 19746 entries, 0 to 19745
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Time                    19746 non-null  str    
 1   Date                    19746 non-null  str    
 2   Sender_account          19746 non-null  int64  
 3   Receiver_account        19746 non-null  int64  
 4   Amount                  19746 non-null  float64
 5   Payment_currency        19746 non-null  str    
 6   Received_currency       19746 non-null  str    
 7   Sender_bank_location    19746 non-null  str    
 8   Receiver_bank_location  19746 non-null  str    
 9   Payment_type            19746 non-null  str    
 10  Is_laundering           19746 non-null  int64  
dtypes: float64(1), int64(3), str(7)
memory usage: 2.6 MB


### Extracting the hour from 'Time' feature

In [24]:
df_combined['Hour'] = df_combined['Time'].str[:2].astype(int)

In [25]:
df_combined['Date'] = pd.to_datetime(df_combined['Date'])

In [26]:
df_combined['Month'] = df_combined['Date'].dt.month

In [27]:
df_combined['Day_of_week'] = df_combined['Date'].dt.dayofweek

In [28]:
df_combined.head(2)

,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Hour,Month,Day_of_week
0,07:40:08,2023-07-14,3187084517,8846937355,3788.78,UK pounds,UK pounds,UK,UK,Cash Deposit,1,7,7,4
1,22:04:14,2022-10-29,4851027632,9832274459,7377.62,UK pounds,Euro,UK,France,Cross-border,0,22,10,5


In [29]:
df_combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 19746 entries, 0 to 19745
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Time                    19746 non-null  str           
 1   Date                    19746 non-null  datetime64[us]
 2   Sender_account          19746 non-null  int64         
 3   Receiver_account        19746 non-null  int64         
 4   Amount                  19746 non-null  float64       
 5   Payment_currency        19746 non-null  str           
 6   Received_currency       19746 non-null  str           
 7   Sender_bank_location    19746 non-null  str           
 8   Receiver_bank_location  19746 non-null  str           
 9   Payment_type            19746 non-null  str           
 10  Is_laundering           19746 non-null  int64         
 11  Hour                    19746 non-null  int64         
 12  Month                   19746 non-null  int32         
 1

### Removing columns not required for this model 

In [30]:
X_svm = df_combined.drop(columns = ['Time', 'Date', 'Sender_account', 'Receiver_account', 'Is_laundering'])

In [31]:
y_svm = df_combined['Is_laundering']

## Treating the categorical values in X dataset with One Hot Encoding

In [32]:
X_svm.head(2)

,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Hour,Month,Day_of_week
0,3788.78,UK pounds,UK pounds,UK,UK,Cash Deposit,7,7,4
1,7377.62,UK pounds,Euro,UK,France,Cross-border,22,10,5


In [33]:
X_svm_encoded = pd.get_dummies(X_svm, columns=['Payment_currency', 'Received_currency', 
                                               'Sender_bank_location', 'Receiver_bank_location', 'Payment_type'],
                               drop_first=True).astype(int)

## Splitting the dataset into train and test datasets

In [34]:
from sklearn.model_selection import train_test_split

In [35]:
X_train, X_test, y_train, y_test = train_test_split(X_svm_encoded, y_svm, test_size=0.20, stratify=y_svm, random_state=42)

### Treating the bigger values of the 'Amount' Feature

In [36]:
from sklearn.preprocessing import RobustScaler

In [37]:
rc = RobustScaler()

In [38]:
X_train_svm_scaled = rc.fit_transform(X_train)

In [39]:
X_test_svm_scaled = rc.transform(X_test)

## Model 4 : Support Vector Machine

In [43]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC

In [44]:
base_svm = SVC(random_state=42)

In [45]:
model_svm = CalibratedClassifierCV(base_svm, cv=3) 

In [46]:
model_svm.fit(X_train_svm_scaled, y_train)

,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2",SVC(random_state=42)
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'sigmoid'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary <n_jobs>` for more details... versionadded:: 0.24",None
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",'auto'
Name,Type,Value
"calibrated_classifiers_ calibrated_classifiers_: list (len() equal to cv or 1 if `ensemble=False`)The list of classifier and calibrator pairs.- When `ensemble=True`, `n_cv` fitted `estimator` and calibrator pairs. `n_cv` is the number of cross-validation folds.- When `ensemble=False`, the `estimator`, fitted on all the data, and fitted calibrator... versionchanged:: 0.24 Single calibrated classifier case when `ensemble=False`.",list,"[<sklearn.cali...001B979FD16A0>, <sklearn.cali

In [50]:
y_pred_svm = model_svm.predict(X_test_svm_scaled)

## Model Evaluation

In [51]:
from sklearn.metrics import classification_report

In [52]:
print(classification_report(y_test, y_pred_svm))

              precision    recall  f1-score   support

           0       0.69      0.84      0.75      1975
           1       0.79      0.62      0.69      1975

    accuracy                           0.73      3950
   macro avg       0.74      0.73      0.72      3950
weighted avg       0.74      0.73      0.72      3950

